# Tutorial 01: Running an end-to-end pulsar population

We can perform the simulation of a Galactic population of neutron stars in different ways.

In this tutorial, we will focus on the approach that consists of simulating the full population in one go. This means that we have to 
initialize the entire population formed of N neutron stars from some initial conditions, evolve it in time and finally apply the survey models to select those neutron stars that are detected.

For this approach, we run the script `mlpoppyns/simulator/simulate_population_full.py`.
Detailed information about the arguments that we can pass to the script is obtained by issuing the `--help` 
argument, e.g., running:
```
python mlpoppyns/simulator/simulate_population_full.py --help
```
The default input parameters for the simulation are specified in the `mlpoppyns/simulator/config_simulator.py` file.
To simulate populations with different initial parameters, the user can directly modify the simulator configuration in 
`mlpoppyns/simulator/config_simulator.py`.

Alternatively, we can provide a JSON dictionary containing configuration overrides for the various simulation parameters. 
We specify this as a command line argument to the simulator script as follows:
```
python mlpoppyns/simulator/simulate_population_full.py --save_dir output/sim_full --parameter_override parameter_override.json
```
For example, we can set the number of neutron stars to simulate, the kick-velocity model, the parameters of the initial 
distributions of spin periods and magnetic fields, and several other parameters.
The above command will then generate a new directory `output/sim_full` if it does not exist, in which the simulation 
results will be saved.

The output consists of the following files:
* `initial_population.pkl.gz` containing the initial neutron star properties.
* `final_population.pkl.gz` containing the final population properties.
* `.pkl.gz` files for each of the modelled surveys (containing the stars detected by that survey).
* `.json` and `.log` files containing the timing profiles for the simulation, if enabled.
* `configuration.json` containing the configuration parameters for reproducibility.

In [ ]:
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D


def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)


import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.stellar_dynamics.coordinate_conversions as cc
from mlpoppyns.simulator.simulate_population_full import simulate_population

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the end-to-end simulation for a radio pulsar population

We consider the following three radio surveys:

1. the Parks Multibeam Pulsar Survey (PMPS; [Manchester et al. 2001](https://ui.adsabs.harvard.edu/abs/2001MNRAS.328...17M/abstract), [Lorimer et al. 2006](https://ui.adsabs.harvard.edu/abs/2006MNRAS.372..777L/abstract)),
2. the Swinburne Parkes Multibeam Pulsar Survey (SMPS; [Edwards et al. 2001](https://ui.adsabs.harvard.edu/abs/2001MNRAS.326..358E/abstract), [Jacoby et al. 2009](https://ui.adsabs.harvard.edu/abs/2009ApJ...699.2009J/abstract)),
3. the mid- and low-latitude High Time Resolution Universe survey (HTRU; [Keith et al. 2010](https://ui.adsabs.harvard.edu/abs/2010MNRAS.409..619K/abstract)).

We can adjust several parameters in the imported configuration file.
For example, we can change the following:
1. `NS_number`: number of neutron stars to simulate.
2. `t_age_max`: the maximum age for the simulated neutron stars in [yr].
3. `B_initial_log10_mean` and `B_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial magnetic field.
4. `P_initial_log10_mean` and `P_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial spin period.
5. `a_late`: the power-law index for the late time decay of the magnetic field after $\sim 10^6$ yr.

NOTE: To obtain a realistic birth rate ($\sim 1$ neutron star per century) and a reasonable observed pulsar population, we need to set `NS_number` and `t_age_max` accordingly. However, by increasing both `NS_number` and `t_age_max` the simulation will take more time to run.

We will define the following simulation parameters:

In [ ]:
cfg["NS_number"] = 10000
cfg["t_age_max"] = 3.0e7
cfg["B_initial_log10_mean"] = 13.1
cfg["B_initial_log10_sigma"] = 0.45
cfg["P_initial_log10_mean"] = -1.0
cfg["P_initial_log10_sigma"] = 0.38
cfg["a_late"] = -1.80

Alternatively, we can directly change the parameters in the `parameter_override.json` file in the `tutorials/tutorial_notebooks` folder and pass it to the simulator. Note that we will not use this in the example below, however.

In [ ]:
override_dir = "parameter_override.json"

We next specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_full"

We can now run the simulation by calling the `simulate_population` function from the `mlpoppyns.simulator.simulate_population_full` module with our specific parameter choices.

WARNING: If a `FileNotFound` error appears, remember to check the path to the repository in the `mlpoppyns/simulator/config_simulator.json` configuration file as outlined in the GitHub `README.md` and the `Getting started` page of our documentation.

In [ ]:
simulation_args = argparse.Namespace(
    save_dir=output_dir,
    parameter_override=None,
)
simulate_population(simulation_args)

### Reading the simulation results

We first read in our compressed `.pkl` files.

In [ ]:
data_full = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "final_population.pkl.gz"),
    compression="gzip",
)
data_full.columns

In [ ]:
data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.columns

In [ ]:
data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.columns

In [ ]:
data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_low_mid_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_low_mid.columns

In [ ]:
data_HTRU_high = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_high_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_high.columns

Extracting the parameters of the full population.

In [ ]:
r = data_full["r"]["[kpc]"].to_numpy()
phi = data_full["phi"]["[rad]"].to_numpy()
x = r * np.cos(phi)
y = r * np.sin(phi)
z = data_full["z"]["[kpc]"].to_numpy()
ra = data_full["ra"]["[deg]"].to_numpy()
dec = data_full["dec"]["[deg]"].to_numpy()
pm_ra = data_full["pm_ra"]["[mas yr^-1]"].to_numpy()
pm_dec = data_full["pm_dec"]["[mas yr^-1]"].to_numpy()
v_r = data_full["v_r"]["[km s^-1]"].to_numpy()
v_phi = data_full["v_phi"]["[km s^-1]"].to_numpy()
v_z = data_full["v_z"]["[km s^-1]"].to_numpy()
dist = data_full["dist"]["[kpc]"].to_numpy()
B = data_full["B"]["[G]"].to_numpy()
chi = data_full["chi"]["[rad]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()
P_dot = data_full["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = data_full["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = data_full["w_int"]["[s]"].to_numpy()
intercepted_radio = data_full["intercepted_radio"][" "].to_numpy(dtype=bool)
age = data_full["age"]["[yr]"].to_numpy()
l, b, _, _, _, _ = cc.galactocentric_to_galactic(
    x, y, z, np.zeros(len(x)), np.zeros(len(x)), np.zeros(len(x))
)

Extracting the parameters of detected pulsars in the individual surveys.

In [ ]:
idx_PMPS = data_PMPS["idx"].to_numpy(dtype=int)
S_radio_PMPS = data_PMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_PMPS = data_PMPS["w_eff"]["[s]"].to_numpy()

idx_SMPS = data_SMPS["idx"].to_numpy(dtype=int)
S_radio_SMPS = data_SMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_SMPS = data_SMPS["w_eff"]["[s]"].to_numpy()

idx_HTRU_low_mid = data_HTRU_low_mid["idx"].to_numpy(dtype=int)
S_radio_HTRU_low_mid = data_HTRU_low_mid["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_low_mid = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

idx_HTRU_high = data_HTRU_high["idx"].to_numpy(dtype=int)
S_radio_HTRU_high = data_HTRU_high["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_high = data_HTRU_high["w_eff"]["[s]"].to_numpy()

In [ ]:
set_PMPS = set(idx_PMPS.tolist())
set_SMPS = set(idx_SMPS.tolist())
set_HTRU_low_mid = set(idx_HTRU_low_mid.tolist())
set_HTRU_high = set(idx_HTRU_high.tolist())

idx_radio_detected_noduplicates = list(
    set_PMPS | set_SMPS | set_HTRU_low_mid | set_HTRU_high
)

Extracting the detection fractions.

In [ ]:
number_intercepted = len(intercepted_radio[intercepted_radio == True])
number_detected_PMPS = len(idx_PMPS)
number_detected_SMPS = len(idx_SMPS)
number_detected_HTRU_low_mid = len(idx_HTRU_low_mid)
number_detected_HTRU_high = len(idx_HTRU_high)
number_detected_total = len(idx_radio_detected_noduplicates)

fraction_intercepted = len(intercepted_radio[intercepted_radio == True]) / len(
    intercepted_radio
)
fraction_detected_PMPS = len(idx_PMPS) / len(intercepted_radio)
fraction_detected_SMPS = len(idx_SMPS) / len(intercepted_radio)
fraction_detected_HTRU_low_mid = len(idx_HTRU_low_mid) / len(intercepted_radio)
fraction_detected_HTRU_high = len(idx_HTRU_high) / len(intercepted_radio)

In [ ]:
print(f"Total number of detected pulsars: {number_detected_total}")
print(
    f"Fraction of pulsars pointing at us: {fraction_intercepted}, ({number_intercepted}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by PMPS: {fraction_detected_PMPS}, ({number_detected_PMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by SMPS: {fraction_detected_SMPS}, ({number_detected_SMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU low and mid latitude: {fraction_detected_HTRU_low_mid}, ({number_detected_HTRU_low_mid}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU high latitude: {fraction_detected_HTRU_high}, ({number_detected_HTRU_high}/{len(intercepted_radio)})"
)

### Plotting the simulation results

As a first diagnostic, we plot our population and corresponding detection in Galactic longitude and latitude.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l,
    b,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label=r"Simulation all",
)
ax.plot(
    l[intercepted_radio],
    b[intercepted_radio],
    linestyle="None",
    marker="o",
    color="gray",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label=r"Intercepting our LOS",
)
ax.plot(
    l[idx_PMPS],
    b[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=7,
    alpha=1,
    rasterized=True,
    label=r"Detected by PMPS",
)
ax.plot(
    l[idx_SMPS],
    b[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Detected by SMPS",
)
ax.plot(
    l[idx_HTRU_low_mid],
    b[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Detected by HTRU",
)

ax.plot(
    0.0,
    0.0,
    linestyle="None",
    marker="*",
    color="tab:orange",
    markersize=20,
    label="Galactic center",
)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
plt.legend(
    bbox_to_anchor=(1, 1),
    frameon=False,
    loc=0,
    fontsize=20,
    markerscale=5,
    handler_map={
        PathCollection: HandlerPathCollection(update_func=update),
        plt.Line2D: HandlerLine2D(update_func=update),
    },
)
plt.show()

We can also visualize our population and corresponding detections in the $P-\dot{P}$ plane.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    P_dot,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label="Simulation all",
)
ax.plot(
    P[intercepted_radio],
    P_dot[intercepted_radio],
    linestyle="None",
    marker="o",
    color="gray",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    P[idx_PMPS],
    P_dot[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=7,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    P_dot[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    P_dot[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)

ax.set_xscale("log")
ax.set_yscale("log")

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(
    bbox_to_anchor=(1, 1),
    frameon=False,
    loc=0,
    fontsize=20,
    markerscale=5,
    handler_map={
        PathCollection: HandlerPathCollection(update_func=update),
        plt.Line2D: HandlerLine2D(update_func=update),
    },
)

plt.show()

## Setting up and running the end-to-end simulation for a radio and X-ray neutron star population

In order to include the simulation of the neutron star population in the X-ray band, we need to set the parameter `xray_simulation` to `True`.

We consider two all-sky X-ray survey setups:

1. a simple flux limited survey with a threshold flux, i.e., all neutron star with X-ray flux greater than this value are detected;
2. a more realistic survey prescription that takes into account two threshold fluxes sampled from two Gaussian distributions centred at $10^{-14}$ and $10^{-12}$ erg s$^{-1}$ cm$^{-2}$, respectively. The lower flux threshold mimics a more sensitive survey like the one performed during targeted sources (e.g., magnetars that underwent outbursts); the higher flux threshold mimics the sensitivity of all-sky surveys able to detect only the brightest objects (see Ronchi et al. 2026 for more details).

As before, we can adjust several parameters in the imported configuration file. For example, we can change the following:

1. `NS_number`: number of neutron stars to simulate.
2. `t_age_max`: the maximum age for the simulated neutron stars in [yr].
3. `magnetic_field_model`: the model for the distribution of birth magnetic fields.
4. `B_initial_log10_mean_comp1`, `B_initial_log10_sigma_comp1`, `B_initial_log10_mean_comp2`, `B_initial_log10_sigma_comp2`, `B_initial_log10_weight_comp1`: the means, standard deviations and weight of the first component of the double Gaussian distribution for the $\log_{10}$ of the initial magnetic field.
5. `P_initial_log10_mean` and `P_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial spin period.
6. `a_late`: the power-law index for the late time decay of the magnetic field after $\sim 10^6$ yr.

NOTE: To obtain a realistic birth rate ($\sim 1$ neutron star per century) and a reasonable observed pulsar population, we need to set `NS_number` and `t_age_max` accordingly. However, by increasing both `NS_number` and `t_age_max` the simulation will take more time to run.

We will define the following simulation parameters:

In [ ]:
cfg["simulation_xray"] = True
cfg["NS_number"] = 10000
cfg["t_age_max"] = 3.0e7
cfg["magnetic_field_model"] = "double_log-normal"
cfg["B_initial_log10_mean_comp1"] = 12.7
cfg["B_initial_log10_sigma_comp1"] = 0.5
cfg["B_initial_log10_mean_comp2"] = 14.1
cfg["B_initial_log10_sigma_comp2"] = 0.3
cfg["B_initial_log10_weight_comp1"] = 0.5
cfg["P_initial_log10_mean"] = -1.0
cfg["P_initial_log10_sigma"] = 0.38
cfg["a_late"] = -1.0

Alternatively, we can directly change the parameters in the `parameter_override.json` file in the `tutorials/tutorial_notebooks` folder and pass it to the simulator. Note that we will not use this in the example below, however.

In [ ]:
override_dir = "parameter_override.json"

We next specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_full_xray"

We can now run the simulation by calling the simulate_population function from the mlpoppyns.simulator.simulate_population_full module with our specific parameter choices.

WARNING: If a FileNotFound error appears, remember to check the path to the repository in the mlpoppyns/simulator/config_simulator.json configuration file as outlined in the GitHub README.md and the Getting started page of our documentation.

In [ ]:
simulation_args = argparse.Namespace(
    save_dir=output_dir,
    parameter_override=None,
)
simulate_population(simulation_args)

### Reading the simulation results

We first read in our compressed `.pkl` files.

In [ ]:
data_full = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "final_population.pkl.gz"),
    compression="gzip",
)
data_full.columns

In [ ]:
data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.columns

In [ ]:
data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.columns

In [ ]:
data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_low_mid_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_low_mid.columns

In [ ]:
data_xray = pd.read_pickle(
    pathlib.Path().joinpath(
        output_dir, "survey_xray_flux_threshold_results.pkl.gz"
    ),
    compression="gzip",
)
data_xray.columns

Extracting the parameters of the full population.

In [ ]:
r = data_full["r"]["[kpc]"].to_numpy()
phi = data_full["phi"]["[rad]"].to_numpy()
x = r * np.cos(phi)
y = r * np.sin(phi)
z = data_full["z"]["[kpc]"].to_numpy()
ra = data_full["ra"]["[deg]"].to_numpy()
dec = data_full["dec"]["[deg]"].to_numpy()
pm_ra = data_full["pm_ra"]["[mas yr^-1]"].to_numpy()
pm_dec = data_full["pm_dec"]["[mas yr^-1]"].to_numpy()
v_r = data_full["v_r"]["[km s^-1]"].to_numpy()
v_phi = data_full["v_phi"]["[km s^-1]"].to_numpy()
v_z = data_full["v_z"]["[km s^-1]"].to_numpy()
dist = data_full["dist"]["[kpc]"].to_numpy()
B = data_full["B"]["[G]"].to_numpy()
chi = data_full["chi"]["[rad]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()
P_dot = data_full["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = data_full["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = data_full["w_int"]["[s]"].to_numpy()
intercepted_radio = data_full["intercepted_radio"][" "].to_numpy(dtype=bool)
L_x_therm = data_full["L_x_therm"]["[erg s^-1]"].to_numpy(dtype=bool)
S_x_rcs_abs = data_full["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy(dtype=bool)
outburst = data_full["outburst"][" "].to_numpy(dtype=bool)
age = data_full["age"]["[yr]"].to_numpy()
l, b, _, _, _, _ = cc.galactocentric_to_galactic(
    x, y, z, np.zeros(len(x)), np.zeros(len(x)), np.zeros(len(x))
)

Extracting the parameters of detected pulsars in the individual surveys.

In [ ]:
idx_PMPS = data_PMPS["idx"].to_numpy(dtype=int)
S_radio_PMPS = data_PMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_PMPS = data_PMPS["w_eff"]["[s]"].to_numpy()

idx_SMPS = data_SMPS["idx"].to_numpy(dtype=int)
S_radio_SMPS = data_SMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_SMPS = data_SMPS["w_eff"]["[s]"].to_numpy()

idx_HTRU_low_mid = data_HTRU_low_mid["idx"].to_numpy(dtype=int)
S_radio_HTRU_low_mid = data_HTRU_low_mid["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_low_mid = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

idx_xray = data_xray["idx"].to_numpy(dtype=int)
S_xray = data_xray["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy()

In [ ]:
set_PMPS = set(idx_PMPS.tolist())
set_SMPS = set(idx_SMPS.tolist())
set_HTRU_low_mid = set(idx_HTRU_low_mid.tolist())

idx_radio_detected_noduplicates = list(set_PMPS | set_SMPS | set_HTRU_low_mid)

Extracting the detection fractions.

In [ ]:
number_intercepted = len(intercepted_radio[intercepted_radio == True])
number_detected_PMPS = len(idx_PMPS)
number_detected_SMPS = len(idx_SMPS)
number_detected_HTRU_low_mid = len(idx_HTRU_low_mid)
number_detected_xray = len(idx_xray)
number_detected_total = len(idx_radio_detected_noduplicates)

fraction_intercepted = len(intercepted_radio[intercepted_radio == True]) / len(
    intercepted_radio
)
fraction_detected_PMPS = len(idx_PMPS) / len(intercepted_radio)
fraction_detected_SMPS = len(idx_SMPS) / len(intercepted_radio)
fraction_detected_HTRU_low_mid = len(idx_HTRU_low_mid) / len(intercepted_radio)
fraction_detected_xray = len(idx_xray) / len(intercepted_radio)

In [ ]:
print(f"Total number of detected neutron stars: {number_detected_total}")
print(
    f"Fraction of pulsars pointing at us: {fraction_intercepted}, ({number_intercepted}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by PMPS: {fraction_detected_PMPS}, ({number_detected_PMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by SMPS: {fraction_detected_SMPS}, ({number_detected_SMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU low and mid latitude: {fraction_detected_HTRU_low_mid}, ({number_detected_HTRU_low_mid}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by X-ray survey: {fraction_detected_xray}, ({number_detected_xray}/{len(intercepted_radio)})"
)

### Plotting the simulation results

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l,
    b,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label=r"Simulation all",
)
ax.plot(
    l[intercepted_radio],
    b[intercepted_radio],
    linestyle="None",
    marker="o",
    color="gray",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label=r"Intercepting our LOS",
)
ax.plot(
    l[idx_PMPS],
    b[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=7,
    alpha=1,
    rasterized=True,
    label=r"Detected by PMPS",
)
ax.plot(
    l[idx_SMPS],
    b[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Detected by SMPS",
)
ax.plot(
    l[idx_HTRU_low_mid],
    b[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Detected by HTRU",
)
ax.plot(
    l[idx_xray],
    b[idx_xray],
    linestyle="None",
    marker="X",
    # fillstyle="none",
    color="tab:green",
    markersize=10,
    alpha=1,
    rasterized=True,
    label=r"Detected in X-ray",
)

ax.plot(
    0.0,
    0.0,
    linestyle="None",
    marker="*",
    color="tab:orange",
    markersize=20,
    label="Galactic center",
)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
plt.legend(
    bbox_to_anchor=(1, 1),
    frameon=False,
    loc=0,
    fontsize=20,
    markerscale=5,
    handler_map={
        PathCollection: HandlerPathCollection(update_func=update),
        plt.Line2D: HandlerLine2D(update_func=update),
    },
)
plt.show()

We can also visualize our population and corresponding detections in the $P-\dot{P}$ plane.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    P_dot,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label="Simulation all",
)
ax.plot(
    P[intercepted_radio],
    P_dot[intercepted_radio],
    linestyle="None",
    marker="o",
    color="gray",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    P[idx_PMPS],
    P_dot[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=7,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    P_dot[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    P_dot[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    P[idx_xray],
    P_dot[idx_xray],
    linestyle="None",
    marker="X",
    # fillstyle="none",
    color="tab:green",
    markersize=10,
    alpha=1.0,
    rasterized=True,
    label="Detected in X-ray",
)

ax.set_xscale("log")
ax.set_yscale("log")

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(
    bbox_to_anchor=(1, 1),
    frameon=False,
    loc=0,
    fontsize=20,
    markerscale=5,
    handler_map={
        PathCollection: HandlerPathCollection(update_func=update),
        plt.Line2D: HandlerLine2D(update_func=update),
    },
)

plt.show()